In [67]:
import jax.numpy as jnp, jax

from untangle.utils import function_error, cpd_error, collect_information
from untangle.algorithm import cmtf_bsd
from untangle.scaler import JacobianScaler, FunctionScaler
from untangle._common import default_dof
from untangle._ops import reconstruct

key = jax.random.key(0)

In [68]:
def f(x):
    x1, x2 = x
    return jnp.array([
        jnp.cos(3*jnp.pi*x1 + x2) + x2**2,
        -jnp.sin(2*jnp.pi*x2 + x1) + x1**3,
    ])
def fscaled100(x):
    x1, x2 = x
    return jnp.array([
        100*(jnp.cos(3*jnp.pi*x1) + x2**2),
        -jnp.sin(2*jnp.pi*x2) + x1**3,
    ])

In [69]:
K = 20
N = 50
rank = 4
dof = default_dof(N) - int(default_dof(N) % 2 == 0)
print(dof)

13


In [70]:
help(cpd_error)

Help on function cpd_error in module untangle.utils.utils:

cpd_error(
    tensor: Float[Array, 'n m N'],
    factors: Tuple[Float[Array, 'n r'], Float[Array, 'm r'], Float[Array, 'N r']],
    weights: Optional[Float[Array, 'r']] = None
) -> Float[Array, '']



In [71]:
for func in [f, fscaled100]:
    e1 = e2 = te = 0
    e1_fs = e2_fs = te_fs = 0
    e1_ts = e2_ts = te_ts = 0

    for k in jax.random.split(key, K):

        fscaler = FunctionScaler(func, 2, key=k)
        func_scaled = fscaler.scale()
        
        X, Y, J = collect_information(func, N, key=k)
        X, Y_fs, J_fs = collect_information(func_scaled, N, key=k)
        
        tscaler = JacobianScaler(J, Y)
        J_scaled, Y_scaled = tscaler.scale()
        
        decoupling = cmtf_bsd(X, Y, J, rank, dof=dof, key=k, niters=50)

        decoupling_fscaled = cmtf_bsd(X, Y_fs, J_fs, rank, dof=dof, key=k, niters=50)

        decoupling_tscaled = cmtf_bsd(X, Y_scaled, J_scaled, rank, dof=dof, key=k, niters=50)

        errors = function_error(func, decoupling, X)
        errors_ts = function_error(func, tscaler.unscale(decoupling_tscaled), X)
        errors_fs = function_error(func, fscaler.unscale(decoupling_fscaled), X)

        #print(errors)
        #print(errors_fs)
        #print(errors_ts)

        T_ts = reconstruct(*decoupling_tscaled.factors[:-1], jnp.ones(rank))
        T_fs = reconstruct(*decoupling_fscaled.factors[:-1], jnp.ones(rank))

        T_fs = T_fs / fscaler.factors[:, None, None]
        T_ts = T_ts / tscaler.factors[:, None, None]

        te_ts += jnp.linalg.norm(J - T_ts) / jnp.linalg.norm(J)
        te_fs += jnp.linalg.norm(J - T_fs) / jnp.linalg.norm(J)
        
        te += cpd_error(J, decoupling.factors[:-1])
        e1 += errors[0]
        e2 += errors[1]

        e1_fs += errors_fs[0]
        e2_fs += errors_fs[1]

        e1_ts += errors_ts[0]
        e2_ts += errors_ts[1]

    print(te / K)
    print(te_fs / K)
    print(te_ts / K)
    print(e1 / K, e2 / K)
    print(e1_fs / K, e2_fs / K)
    print(e1_ts / K, e2_ts / K)

|CMTF-BSD| (rank=4): 100%|████████████████████████████████████████████| 50/50 [00:00<00:00, 158.21it/s, error=0.0308, best=0.0283 (32)]
|CMTF-BSD| (rank=4): 100%|████████████████████████████████████████████| 50/50 [00:00<00:00, 174.17it/s, error=0.0220, best=0.0220 (49)]
|CMTF-BSD| (rank=4): 100%|████████████████████████████████████████████| 50/50 [00:00<00:00, 176.24it/s, error=0.0174, best=0.0165 (19)]
|CMTF-BSD| (rank=4): 100%|████████████████████████████████████████████| 50/50 [00:00<00:00, 157.67it/s, error=0.0301, best=0.0298 (44)]
|CMTF-BSD| (rank=4): 100%|████████████████████████████████████████████| 50/50 [00:00<00:00, 169.56it/s, error=0.0300, best=0.0298 (38)]
|CMTF-BSD| (rank=4): 100%|████████████████████████████████████████████| 50/50 [00:00<00:00, 165.31it/s, error=0.0205, best=0.0205 (49)]
|CMTF-BSD| (rank=4): 100%|████████████████████████████████████████████| 50/50 [00:00<00:00, 164.19it/s, error=0.0238, best=0.0237 (45)]
|CMTF-BSD| (rank=4): 100%|██████████████████████

0.024890866
0.026498621
0.03683949
3.009168 2.1856675
3.1660805 2.3027456
3.8377442 2.043131


|CMTF-BSD| (rank=4): 100%|████████████████████████████████████████████| 50/50 [00:00<00:00, 162.14it/s, error=0.0377, best=0.0347 (48)]
|CMTF-BSD| (rank=4): 100%|████████████████████████████████████████████| 50/50 [00:00<00:00, 175.53it/s, error=0.0512, best=0.0512 (46)]
|CMTF-BSD| (rank=4): 100%|████████████████████████████████████████████| 50/50 [00:00<00:00, 173.23it/s, error=0.0221, best=0.0191 (40)]
|CMTF-BSD| (rank=4): 100%|████████████████████████████████████████████| 50/50 [00:00<00:00, 164.90it/s, error=0.2285, best=0.1606 (27)]
|CMTF-BSD| (rank=4): 100%|████████████████████████████████████████████| 50/50 [00:00<00:00, 169.26it/s, error=0.0217, best=0.0217 (49)]
|CMTF-BSD| (rank=4): 100%|████████████████████████████████████████████| 50/50 [00:00<00:00, 171.59it/s, error=0.0110, best=0.0110 (40)]
|CMTF-BSD| (rank=4): 100%|████████████████████████████████████████████| 50/50 [00:00<00:00, 150.90it/s, error=4.6743, best=0.1604 (40)]
|CMTF-BSD| (rank=4): 100%|██████████████████████

0.060610615
0.022700997
0.018722456
7.6666756 53.101635
2.4140637 1.2767701
1.6790158 0.935655


In [41]:
help(FunctionScaler)

Help on class FunctionScaler in module untangle.scaler.scaler:

class FunctionScaler(builtins.object)
 |  FunctionScaler(
 |      f: Callable,
 |      n_inputs: int,
 |      key: Array = None,
 |      scaler: str = 'max',
 |      N: int = 1000
 |  )
 |
 |  Methods defined here:
 |
 |  __init__(
 |      self,
 |      f: Callable,
 |      n_inputs: int,
 |      key: Array = None,
 |      scaler: str = 'max',
 |      N: int = 1000
 |  )
 |      Initialize self.  See help(type(self)) for accurate signature.
 |
 |  scale(self) -> Callable
 |
 |  unscale(self, f_scaled: Callable) -> Callable
 |
 |  ----------------------------------------------------------------------
 |  Data descriptors defined here:
 |
 |  __dict__
 |      dictionary for instance variables
 |
 |  __weakref__
 |      list of weak references to the object



In [42]:
help(JacobianScaler)

Help on class JacobianScaler in module untangle.scaler.scaler:

class JacobianScaler(builtins.object)
 |  JacobianScaler(J: Float[Array, 'n m N'], Y: Float[Array, 'N n'])
 |
 |  Methods defined here:
 |
 |  __init__(self, J: Float[Array, 'n m N'], Y: Float[Array, 'N n'])
 |      Initialize self.  See help(type(self)) for accurate signature.
 |
 |  scale(self) -> Tuple[Float[Array, 'n m N'], Float[Array, 'N n']]
 |
 |  unscale(self, f_scaled: Callable) -> Callable
 |
 |  unscale_output(self, output: Float[Array, 'n']) -> Float[Array, 'n']
 |
 |  ----------------------------------------------------------------------
 |  Data descriptors defined here:
 |
 |  __dict__
 |      dictionary for instance variables
 |
 |  __weakref__
 |      list of weak references to the object

